In [1]:
import importlib
import gymnasium as gym

import env.afm_env
importlib.reload(env.afm_env)
from env.afm_env import AfmEnvironment

# gym.register(
#     id="AfmEnvironment",
#     entry_point=AfmEnvironment,
# )
#
# env = gym.make(
#     "AfmEnvironment",
#     data_file_path="environments/pt_111_small_5row_missing.npz"
# )

 PACKAGE_PATH =  /home/henry/miniforge3/envs/IProject/lib/python3.11/site-packages/ppafm
 CPP_PATH     =  /home/henry/miniforge3/envs/IProject/lib/python3.11/site-packages/ppafm/cpp


In [2]:
import tqdm
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecNormalize
import os
import datetime
from torch.nn import ReLU

TRAIN_BASE_DIR = "train_results"
train_date = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# TRAIN_DIR = os.path.join(TRAIN_BASE_DIR, train_date)

model_name = "sac_afm_model_" + train_date
TRAIN_DIR = os.path.join(TRAIN_BASE_DIR, model_name)
os.makedirs(TRAIN_DIR, exist_ok=True)

def make_env_gpu(rank=0):
    def _init():
        env = AfmEnvironment(
            surface_path="materials/pt_111_small_5row_missing.xyz",
            params_path="materials/params_code.ini",
            num_historic_data=30,
            num_actions=1,
        )
        env = Monitor(env)  #, filename=os.path.join("./logs", f"env_{rank}"))
        return env

    return _init


def make_env_load(rank=0):
    def _init():
        env = AfmEnvironment(
            data_dir_path="environments/pt_111_small_rows_missing",
            num_historic_data=30,
            num_actions=1,
        )
        env = Monitor(env)  #, filename=os.path.join("./logs", f"env_{rank}"))
        return env

    return _init

n_envs = 4
env_arr = [make_env_load(i) for i in range(n_envs)]
vec_env = DummyVecEnv(env_arr)
# vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)

policy_kwargs = dict(
    net_arch=[256, 256],
    activation_fn=ReLU,
)

model = SAC(
    "MultiInputPolicy",
    vec_env,
    verbose=1,
    tensorboard_log="./tensorboard_logs_testing",
    policy_kwargs=policy_kwargs,
    gradient_steps=n_envs,
)

checkpoint_callback = CheckpointCallback(
    save_freq=10000,
    save_path=os.path.join(TRAIN_DIR, "models"),
    name_prefix=model_name,
    save_replay_buffer=False,
)

model.learn(
    total_timesteps=10000000,
    log_interval=1,
    progress_bar=True,
    tb_log_name=model_name,#"sac_afm_env" + train_date,
    callback=[checkpoint_callback],
)
model.save(os.path.join(TRAIN_DIR, "final_model"))
# vec_env.save(os.path.join(TRAIN_DIR, "vec_normalize.pkl"))

Using cuda device
Logging to ./tensorboard_logs_testing/4_envs_10


Output()

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 8        |
|    ep_rew_mean     | -43.8    |
| time/              |          |
|    episodes        | 1        |
|    fps             | 2207     |
|    time_elapsed    | 0        |
|    total_timesteps | 32       |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 61.5     |
|    ep_rew_mean     | 162      |
| time/              |          |
|    episodes        | 2        |
|    fps             | 138      |
|    time_elapsed    | 3        |
|    total_timesteps | 492      |
| train/             |          |
|    actor_loss      | -8.5     |
|    critic_loss     | 12.1     |
|    ent_coef        | 0.891    |
|    ent_coef_loss   | -0.183   |
|    learning_rate   | 0.0003   |
|    n_updates       | 388      |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 106      |
|    ep_rew_mean     | 355      |
| time/              |          |
|    episodes        | 3        |
|    fps             | 134      |
|    time_elapsed    | 5        |
|    total_timesteps | 776      |
| train/             |          |
|    actor_loss      | -14.7    |
|    critic_loss     | 10.4     |
|    ent_coef        | 0.823    |
|    ent_coef_loss   | -0.284   |
|    learning_rate   | 0.0003   |
|    n_updates       | 672      |
---------------------------------


...

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.55e+04 |
|    ep_rew_mean     | 1.44e+05 |
| time/              |          |
|    episodes        | 88       |
|    fps             | 121      |
|    time_elapsed    | 11701    |
|    total_timesteps | 1426616  |
| train/             |          |
|    actor_loss      | -899     |
|    critic_loss     | 1.18     |
|    ent_coef        | 0.214    |
|    ent_coef_loss   | -0.0413  |
|    learning_rate   | 0.0003   |
|    n_updates       | 1426512  |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.58e+04 |
|    ep_rew_mean     | 1.46e+05 |
| time/              |          |
|    episodes        | 89       |
|    fps             | 121      |
|    time_elapsed    | 11802    |
|    total_timesteps | 1438796  |
| train/             |          |
|    actor_loss      | -899     |
|    critic_loss     | 6.58     |
|    ent_coef        | 0.197    |
|    ent_coef_loss   | -0.0318  |
|    learning_rate   | 0.0003   |
|    n_updates       | 1438692  |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.56e+04 |
|    ep_rew_mean     | 1.45e+05 |
| time/              |          |
|    episodes        | 90       |
|    fps             | 121      |
|    time_elapsed    | 11814    |
|    total_timesteps | 1440448  |
| train/             |          |
|    actor_loss      | -898     |
|    critic_loss     | 2.42     |
|    ent_coef        | 0.248    |
|    ent_coef_loss   | 0.0903   |
|    learning_rate   | 0.0003   |
|    n_updates       | 1440344  |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.55e+04 |
|    ep_rew_mean     | 1.43e+05 |
| time/              |          |
|    episodes        | 91       |
|    fps             | 121      |
|    time_elapsed    | 11930    |
|    total_timesteps | 1454920  |
| train/             |          |
|    actor_loss      | -900     |
|    critic_loss     | 3.29     |
|    ent_coef        | 0.16     |
|    ent_coef_loss   | 0.225    |
|    learning_rate   | 0.0003   |
|    n_updates       | 1454816  |
---------------------------------


KeyboardInterrupt: 

In [3]:
from matplotlib import pyplot as plt

# env = AfmEnvironment(
#     data_file_path="environments/pt_111_small_rows_missing.npz",
#     num_historic_data=30,
#     num_actions=1,
# )

env = AfmEnvironment(
    surface_path="materials/pt_111_small_5row_missing.xyz",
    params_path="materials/params_code.ini",
    num_historic_data=30,
    num_actions=1,
)

# Manually test environment. Reset, step to where we'll get a proper image and scan the same height
env.reset(123)
env.step([0])
env.step([0])
env.step([0])
env.step([0])
env.step([0])

for i in range(201*201):
    env.step([100])

plt.imshow(env.generated_image.T)
plt.colorbar()
plt.show()



Initializing an OpenCL environment on NVIDIA CUDA


AttributeError: 'PpafmParameters' object has no attribute 'save_to_file'